[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/warmup/NB_Warmup_02_LabelEncoder.ipynb)

# NB Warm-up 02 — Pandas and NumPy: from tables to predictions

**STAT3009 · Recommender Systems**  
**Suggested time:** 15–18 minutes; the final reminder is optional.

> **Recall → Predict → Run → Explain.** Revisit last lecture's data operations, then rebuild a user-mean baseline for today's machine-learning lecture.

By the end, you should be able to:

1. select pandas columns and convert them to NumPy arrays;
2. select an array column and use a Boolean mask to calculate a user mean;
3. use `set`, `np.full`, and integer-array indexing to build predictions;
4. keep train/test ID mappings consistent.

**Classroom route:** quick recall (2 min) → one NumPy example (6 min) → course arrays (3 min) → baseline challenge (5 min) → exit ticket (2 min).

## The workflow

```text
pandas table → select columns → NumPy arrays → mask and average → lookup predictions
```

We first use four already-encoded rating triples so that every array operation can be checked by hand. Then we repeat the workflow on the Netflix course data. Encoded IDs are array positions, not numerical measurements.

## 0. Ninety-second prediction

Decide whether each statement is true or false before running any code.

1. For an array with shape `(4, 2)`, `X[:, 0]` selects the first row.
2. `y[users == 0]` keeps the ratings whose matching user ID is 0.
3. `np.full(4, 3.5)` creates four values, each equal to 3.5.
4. Indexing a lookup vector with `[2, 0, 2]` returns three predictions in that order.

<details>
<summary><b>Reveal the answers</b></summary>

1. **False.** `:` keeps all rows; `0` selects the first column. The first row is `X[0]`.
2. **True.** The Boolean mask selects matching positions in `y`.
3. **True.** `np.full` fills every position with the supplied value.
4. **True.** Integer-array indexing preserves the query order and repeated indices.

</details>

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import LabelEncoder

## 2. NumPy recall with four ratings

These user and movie IDs are **already encoded**. Users 0, 1, and 2 have training ratings; user 3 will appear only in our prediction queries.

`df[['user_id', 'movie_id']]` selects the two input columns; `df['rating']` selects the answers. After `.to_numpy(...)`, `X[:, 0]` is the user column. Keep `X` and `y` in the same row order.

In [ ]:
toy_train = pd.DataFrame({
    'user_id': [0, 1, 0, 2],
    'movie_id': [1, 0, 2, 1],
    'rating': [4, 2, 5, 3],
})

toy_X = toy_train[['user_id', 'movie_id']].to_numpy(dtype=int)
toy_y = toy_train['rating'].to_numpy(dtype=float)

display(toy_train)
print('X:', toy_X.shape, toy_X.dtype)
print('y:', toy_y.shape, toy_y.dtype)
print('user column:', toy_X[:, 0])

### Boolean indexing: select one user's ratings

Read `toy_y[toy_X[:, 0] == 0]` from the inside out:

1. select the user column;
2. compare every user ID with 0 to make a Boolean mask;
3. use that mask to select matching positions in the rating vector;
4. average the selected ratings.

**Predict:** which entries of the mask are `True`, how many ratings are selected, and what is user 0's mean? `mask.sum()` counts `True` entries.

In [ ]:
toy_users = toy_X[:, 0]
user_zero_mask = toy_users == 0
user_zero_ratings = toy_y[user_zero_mask]

print('users:', toy_users)
print('mask: ', user_zero_mask)
print('selected ratings:', user_zero_ratings)
print('number of ratings:', user_zero_mask.sum())
print('user 0 mean:', user_zero_ratings.mean())
print('global mean:', np.mean(toy_y))

# The same row selection in pandas:
display(toy_train.loc[toy_train['user_id'] == 0, ['user_id', 'rating']])

### Build a lookup vector and predict

`set(toy_users)` keeps each observed user once. `np.full(n_users, global_mean)` initializes every lookup position with a fallback; the loop replaces positions for users seen in training.

Finally, an integer array requests predictions from the lookup vector in the supplied order, including repeats. **Predict:** which position keeps the fallback, and what will the four query predictions be?

In [ ]:
observed_users = set(toy_users)
print('observed users:', observed_users)

toy_global_mean = toy_y.mean()
toy_user_means = np.full(4, toy_global_mean)
print('before updates:', toy_user_means)

for user in observed_users:
    toy_user_means[user] = toy_y[toy_users == user].mean()

print('after updates: ', toy_user_means)

toy_query_users = np.array([2, 0, 3, 0], dtype=int)
toy_predictions = toy_user_means[toy_query_users]

print('query users:', toy_query_users)
print('predictions:', toy_predictions)
print('prediction shape:', toy_predictions.shape)

assert np.allclose(toy_predictions, [3.0, 4.5, 3.5, 4.5])

## 3. Transfer to the Netflix course data

Read user and movie IDs as strings, inspect the table, and build the same arrays. This time the raw IDs need a consistent mapping to integer positions.

In [ ]:
BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/netflix'
)

COLUMNS = ['user_id', 'movie_id', 'rating']
ID_TYPES = {'user_id': 'string', 'movie_id': 'string'}

train_raw = pd.read_csv(
    f'{BASE_URL}/train.csv',
    usecols=COLUMNS,
    dtype=ID_TYPES,
)[COLUMNS]

test_raw = pd.read_csv(
    f'{BASE_URL}/test.csv',
    usecols=COLUMNS,
    dtype=ID_TYPES,
)[COLUMNS]

In [ ]:
print('train shape:', train_raw.shape)
print('test shape: ', test_raw.shape)
display(train_raw.head())
display(train_raw.dtypes)

### Quick recall: one shared vocabulary per ID column

In this course setting, the test user–movie pairs are available. Concatenate train and test **ID columns**, fit one encoder for users and another for movies, and use each encoder to transform both splits. This assigns positions to test-only IDs without using test ratings.

Before running, explain why we need the **same user mapping** in both tables and a **separate movie mapping**. A short optional reminder appears at the end.

In [ ]:
all_ids = pd.concat(
    [
        train_raw[['user_id', 'movie_id']],
        test_raw[['user_id', 'movie_id']],
    ],
    ignore_index=True,
)

user_encoder = LabelEncoder().fit(all_ids['user_id'])
item_encoder = LabelEncoder().fit(all_ids['movie_id'])

train = train_raw.copy()
test = test_raw.copy()

train['user_id'] = user_encoder.transform(train_raw['user_id'])
test['user_id'] = user_encoder.transform(test_raw['user_id'])

train['movie_id'] = item_encoder.transform(train_raw['movie_id'])
test['movie_id'] = item_encoder.transform(test_raw['movie_id'])

display(train.head())

In [ ]:
print('number of users:', len(user_encoder.classes_))
print('number of movies:', len(item_encoder.classes_))

first_raw_user = train_raw.loc[0, 'user_id']
first_user_index = train.loc[0, 'user_id']
recovered_user = user_encoder.inverse_transform([first_user_index])[0]

print('raw → encoded → recovered')
print(first_raw_user, '→', first_user_index, '→', recovered_user)

assert recovered_user == first_raw_user

### Put the data into a common `X` / `y` format

For the rest of today's class, we will use the same array format throughout:

- `X_train`: two columns containing the training user–movie pairs;
- `y_train`: one rating for each row of `X_train`;
- `X_test`: test user–movie pairs in the same two-column format;
- `y_test_demo`: the corresponding classroom answers, kept separate from prediction.

Row `j` of `X_train` and position `j` of `y_train` must describe the same observed rating.

In [ ]:
X_train = train[['user_id', 'movie_id']].to_numpy(dtype=int)
y_train = train['rating'].to_numpy(dtype=float)

X_test = test[['user_id', 'movie_id']].to_numpy(dtype=int)
y_test_demo = test['rating'].to_numpy(dtype=float)

print('X_train:', X_train.shape, X_train.dtype)
print('y_train:', y_train.shape, y_train.dtype)
print('X_test: ', X_test.shape, X_test.dtype)
print('y_test_demo:', y_test_demo.shape, y_test_demo.dtype)

In [ ]:
assert X_train.shape == (len(train), 2)
assert y_train.shape == (len(train),)
assert X_test.shape == (len(test), 2)
assert y_test_demo.shape == (len(test),)
assert X_train[:, 0].max() < len(user_encoder.classes_)
assert X_train[:, 1].max() < len(item_encoder.classes_)
assert X_test[:, 0].max() < len(user_encoder.classes_)
assert X_test[:, 1].max() < len(item_encoder.classes_)

print('All array checks passed.')

## 4. Your turn: rebuild the user-mean baseline

Use the NumPy patterns just recalled. Spend about four minutes filling the gaps, then explain one line to a neighbour.

1. calculate the global training mean;
2. initialize one array position per encoded user with that fallback;
3. use `set(train_users)` to loop over observed users;
4. select each user's ratings with a Boolean mask and store their mean;
5. index the lookup vector with the user column of `X_test`.

Before running, state the expected shape of `y_pred_user`. For an item-mean baseline, which input column would you select, and how many positions would its lookup vector need?

In [ ]:
# global_mean = ...
# user_mean_vec = np.full(..., global_mean)
# train_users = X_train[:, ...]
#
# for user in set(...):
#     user_mask = ...
#     user_mean_vec[user] = ...
#
# y_pred_user = user_mean_vec[...]
# print(y_pred_user.shape)

### Solution

In [ ]:
global_mean = y_train.mean()
user_mean_vec = np.full(
    len(user_encoder.classes_), global_mean
)
train_users = X_train[:, 0]

for user in set(train_users):
    user_mean_vec[user] = np.mean(
        y_train[train_users == user]
    )

y_pred_user = user_mean_vec[X_test[:, 0]]

print('global mean:', round(global_mean, 3))
print('prediction shape:', y_pred_user.shape)
print('first five predictions:', np.round(y_pred_user[:5], 3))

The arrays now connect both lectures: `LabelEncoder` supplies stable positions, and the user-mean baseline fills those positions with predictions.

## Exit ticket

Answer without running more code:

1. Explain `y_train[X_train[:, 0] == user].mean()` from the inside out.
2. Why use `set(train_users)` and initialize the lookup vector with `np.full(...)`?
3. Why must the same fitted user encoder transform train and test IDs?

**Next:** today's lecture will reuse the same `X` / `y` arrays and prediction logic.

## Compact pandas / NumPy reference

| Goal | Pattern |
|---|---|
| Inspect a table | `df.head()`, `df.shape`, `df.dtypes` |
| Select columns and make arrays | `df[['user_id', 'movie_id']].to_numpy(dtype=int)` |
| Make a rating vector | `df['rating'].to_numpy(dtype=float)` |
| Inspect an array / select a column | `X.shape`, `X.dtype`, `X[:, 0]` |
| Build a Boolean mask | `mask = X[:, 0] == user` |
| Select and average matching ratings | `y[mask].mean()` |
| Find observed IDs | `set(users)` |
| Initialize fallback values | `np.full(n_users, global_mean)` |
| Look up one prediction per row | `user_means[X_test[:, 0]]` |
| Combine ID columns | `pd.concat([...], ignore_index=True)` |
| Fit and apply a shared ID mapping | `encoder.fit(ids)`, `encoder.transform(ids)` |

The optional reminder below is outside the main classroom route.

## Optional reminder: keep ID mappings consistent

Fit one user encoder on the available train/test **ID columns**, then use that same object to transform both tables. A train-only encoder cannot represent a test-only ID, while separate train and test encoders may assign different integers to the same raw ID.

Users and movies still need separate encoders because they are different categorical vocabularies. The encoder uses ID columns only; ratings do not enter this step.